In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from datasets import load_dataset
from tqdm.auto import tqdm
import os

# -- Шаг 0: Конфигурация --
class Config:
    MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
    DATASET_NAME = "juyoungml/HelpSteer2-binarized"
    OUTPUT_DIR = "/trinity/home/a.anokhin/test_rl/test_task/probabilistic_rm_model"
    LEARNING_RATE = 5e-5
    NUM_EPOCHS = 1
    BATCH_SIZE = 8  # Уменьшите, если не хватает VRAM
    MAX_LENGTH = 2048 # Максимальная длина последовательности для токенизатора

# -- Шаг 1: Реализация функции потерь --
# Ваша реализация, обернутая в функцию для чистоты кода.
def compute_probabilistic_reward_loss(logits_w, logits_l):
    """
    Вычисляет функцию потерь для вероятностной RM.
    Loss = -log(P(R_w > R_l))
    """
    # Превращаем логиты в вероятностные распределения
    probs_w = F.softmax(logits_w, dim=-1)
    probs_l = F.softmax(logits_l, dim=-1)

    # Вычисляем кумулятивные вероятности P(R_l <= i)
    cdf_l = torch.cumsum(probs_l, dim=-1)

    # Вычисляем P(R_l < i) путем сдвига кумулятивной функции
    # P(R_l < 1) = 0
    # P(R_l < i) = P(R_l <= i-1) для i > 1
    p_l_less_than_i = torch.cat([
        torch.zeros_like(cdf_l[:, :1]), # P(R_l < 1) = 0
        cdf_l[:, :-1]                  # P(R_l < i) для i=2..10
    ], dim=-1)

    # Итоговая вероятность P(R_w > R_l) = sum_{i=1 to 10} [ P(R_w = i) * P(R_l < i) ]
    prob_win_over_loss = torch.sum(probs_w * p_l_less_than_i, dim=-1)

    # Функция потерь - отрицательный логарифм средней вероятности по батчу
    # Добавляем epsilon для численной стабильности
    loss = -torch.log(prob_win_over_loss + 1e-8).mean()
    
    return loss

# -- Шаг 2.1: Data Collator --
class ProbabilisticRewardDataCollator:
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        chosen_texts = [item['chosen'] for item in batch]
        rejected_texts = [item['rejected'] for item in batch]

        # Токенизируем "winner" тексты
        tokenized_w = self.tokenizer(
            chosen_texts,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Токенизируем "loser" тексты
        tokenized_l = self.tokenizer(
            rejected_texts,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids_w': tokenized_w['input_ids'],
            'attention_mask_w': tokenized_w['attention_mask'],
            'input_ids_l': tokenized_l['input_ids'],
            'attention_mask_l': tokenized_l['attention_mask'],
        }


In [2]:
# -- Основной скрипт обучения --
config = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -- Шаг 2.2: Загрузка модели и токенизатора --
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
# Указываем, что нам нужна голова для классификации с 10 классами (оценки 1-10)
prob_rm_model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=10,
    problem_type="single_label_classification"
).to(device)

# Добавляем pad_token, если его нет
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    prob_rm_model.config.pad_token_id = prob_rm_model.config.eos_token_id

# -- Шаг 2.3: Загрузка и подготовка датасета --
dataset = load_dataset(config.DATASET_NAME, split="train")
# Для демонстрации возьмем небольшую часть датасета
# dataset = dataset.select(range(2000)) 

data_collator = ProbabilisticRewardDataCollator(tokenizer, config.MAX_LENGTH)
train_dataloader = DataLoader(
    dataset,
    batch_size=config.BATCH_SIZE,
    collate_fn=data_collator,
    shuffle=True
)

# -- Шаг 2.4: Настройка оптимизатора и цикла обучения --
optimizer = AdamW(prob_rm_model.parameters(), lr=config.LEARNING_RATE)
num_training_steps = config.NUM_EPOCHS * len(train_dataloader)
lr_scheduler = get_scheduler(
    name="linear", 
    optimizer=optimizer, 
    num_warmup_steps=0, 
    num_training_steps=num_training_steps
)
scaler = GradScaler() # Для fp16

# -- Шаг 2.5: Цикл обучения --
prob_rm_model.train()
progress_bar = tqdm(range(num_training_steps), desc="Training Probabilistic RM")

for epoch in range(config.NUM_EPOCHS):
    total_loss = 0
    for batch in train_dataloader:
        # Перемещаем данные на GPU
        input_ids_w = batch['input_ids_w'].to(device)
        attention_mask_w = batch['attention_mask_w'].to(device)
        input_ids_l = batch['input_ids_l'].to(device)
        attention_mask_l = batch['attention_mask_l'].to(device)

        optimizer.zero_grad()

        # Используем autocast для смешанной точности (fp16)
        with autocast():
            # Получаем логиты для winner и loser
            outputs_w = prob_rm_model(input_ids=input_ids_w, attention_mask=attention_mask_w)
            logits_w = outputs_w.logits

            outputs_l = prob_rm_model(input_ids=input_ids_l, attention_mask=attention_mask_l)
            logits_l = outputs_l.logits

            # Вычисляем loss
            loss = compute_probabilistic_reward_loss(logits_w, logits_l)

        # Backpropagation с использованием GradScaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()
        
        total_loss += loss.item()
        progress_bar.update(1)
        progress_bar.set_postfix({"loss": loss.item()})
        
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{config.NUM_EPOCHS}, Average Loss: {avg_loss:.4f}")

# -- Шаг 2.6: Сохранение модели --
print("Training finished. Saving model...")
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
prob_rm_model.save_pretrained(config.OUTPUT_DIR)
tokenizer.save_pretrained(config.OUTPUT_DIR)
print(f"Model and tokenizer saved to {config.OUTPUT_DIR}")

Using device: cuda


Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at HuggingFaceTB/SmolLM2-135M-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_1052565/2796475399.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # Для fp16


Training Probabilistic RM:   0%|          | 0/903 [00:00<?, ?it/s]

/tmp/ipykernel_1052565/2796475399.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


KeyboardInterrupt: 

In [1]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from datasets import load_dataset
from tqdm.auto import tqdm
import os
import shutil

# -- Шаг 0: Конфигурация --
class Config:
    MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
    DATASET_NAME = "juyoungml/HelpSteer2-binarized"
    OUTPUT_DIR = "/trinity/home/a.anokhin/test_rl/test_task/probabilistic_rm_model"
    LEARNING_RATE = 5e-5
    NUM_EPOCHS = 1 # Эпоха теперь определяет общую продолжительность обучения
    BATCH_SIZE = 4
    MAX_LENGTH = 2048
    # НОВОЕ: Частота валидации и сохранения в шагах
    EVAL_AND_SAVE_STEPS = 500 
    NUM_CHECKPOINTS_TO_KEEP = 2

# -- Шаг 1: Реализация функции потерь --
def compute_probabilistic_reward_loss(logits_w, logits_l):
    probs_w = F.softmax(logits_w, dim=-1)
    probs_l = F.softmax(logits_l, dim=-1)
    cdf_l = torch.cumsum(probs_l, dim=-1)
    p_l_less_than_i = torch.cat([torch.zeros_like(cdf_l[:, :1]), cdf_l[:, :-1]], dim=-1)
    prob_win_over_loss = torch.sum(probs_w * p_l_less_than_i, dim=-1)
    loss = -torch.log(prob_win_over_loss + 1e-8).mean()
    return loss

# -- Шаг 2.1: Data Collator --
class ProbabilisticRewardDataCollator:
    # ... (код без изменений) ...
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        chosen_texts = [item['chosen'] for item in batch]
        rejected_texts = [item['rejected'] for item in batch]
        tokenized_w = self.tokenizer(
            chosen_texts, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt'
        )
        tokenized_l = self.tokenizer(
            rejected_texts, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt'
        )
        return {
            'input_ids_w': tokenized_w['input_ids'], 'attention_mask_w': tokenized_w['attention_mask'],
            'input_ids_l': tokenized_l['input_ids'], 'attention_mask_l': tokenized_l['attention_mask'],
        }


# -- Основной скрипт обучения --
config = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -- Шаг 2.2: Загрузка модели и токенизатора --
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
prob_rm_model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME, num_labels=10, problem_type="single_label_classification"
).to(device)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    prob_rm_model.config.pad_token_id = prob_rm_model.config.eos_token_id

# -- Шаг 2.3: Загрузка и подготовка датасета --
dataset = load_dataset(config.DATASET_NAME, split="train")
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(eval_dataset)}")

data_collator = ProbabilisticRewardDataCollator(tokenizer, config.MAX_LENGTH)
train_dataloader = DataLoader(
    train_dataset, batch_size=config.BATCH_SIZE, collate_fn=data_collator, shuffle=True
)
eval_dataloader = DataLoader(
    eval_dataset, batch_size=config.BATCH_SIZE, collate_fn=data_collator, shuffle=False
)

# -- Шаг 2.4: Настройка оптимизатора и цикла обучения --
optimizer = AdamW(prob_rm_model.parameters(), lr=config.LEARNING_RATE)
num_training_steps = config.NUM_EPOCHS * len(train_dataloader)
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)
scaler = torch.amp.GradScaler('cuda')
saved_checkpoints = []

# -- Шаг 2.5: Цикл обучения и валидации по шагам --
prob_rm_model.train() # Устанавливаем режим обучения один раз в начале
progress_bar = tqdm(range(num_training_steps), desc="Training Steps")

global_step = 0 # НОВОЕ: Глобальный счетчик шагов

for epoch in range(config.NUM_EPOCHS):
    for batch in train_dataloader:
        input_ids_w = batch['input_ids_w'].to(device)
        attention_mask_w = batch['attention_mask_w'].to(device)
        input_ids_l = batch['input_ids_l'].to(device)
        attention_mask_l = batch['attention_mask_l'].to(device)

        optimizer.zero_grad()
        with autocast(device_type='cuda', dtype=torch.float16):
            outputs_w = prob_rm_model(input_ids=input_ids_w, attention_mask=attention_mask_w)
            logits_w = outputs_w.logits
            outputs_l = prob_rm_model(input_ids=input_ids_l, attention_mask=attention_mask_l)
            logits_l = outputs_l.logits
            loss = compute_probabilistic_reward_loss(logits_w, logits_l)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()
        
        progress_bar.update(1)
        progress_bar.set_postfix({"loss": loss.item()})
        global_step += 1 # НОВОЕ: Инкрементируем счетчик шагов

        # НОВОЕ: Блок валидации и сохранения по шагам
        if global_step > 0 and global_step % config.EVAL_AND_SAVE_STEPS == 0:
            print(f"\n--- Running Validation & Saving Checkpoint at Step {global_step} ---")
            
            # --- Валидационный цикл ---
            prob_rm_model.eval() # Переключаем модель в режим оценки
            total_eval_loss = 0
            total_correct_predictions = 0
            with torch.no_grad():
                for eval_batch in tqdm(eval_dataloader, desc=f"Validating at step {global_step}", leave=False):
                    # ... (логика вычисления loss и accuracy, без изменений) ...
                    input_ids_w_eval = eval_batch['input_ids_w'].to(device)
                    attention_mask_w_eval = eval_batch['attention_mask_w'].to(device)
                    input_ids_l_eval = eval_batch['input_ids_l'].to(device)
                    attention_mask_l_eval = eval_batch['attention_mask_l'].to(device)

                    with autocast(device_type='cuda', dtype=torch.float16):
                        outputs_w = prob_rm_model(input_ids=input_ids_w_eval, attention_mask=attention_mask_w_eval)
                        logits_w = outputs_w.logits
                        outputs_l = prob_rm_model(input_ids=input_ids_l_eval, attention_mask=attention_mask_l_eval)
                        logits_l = outputs_l.logits
                        eval_loss = compute_probabilistic_reward_loss(logits_w, logits_l)
                    total_eval_loss += eval_loss.item()
                    
                    probs_w = F.softmax(logits_w, dim=-1)
                    probs_l = F.softmax(logits_l, dim=-1)
                    cdf_l = torch.cumsum(probs_l, dim=-1)
                    p_l_less_than_i = torch.cat([torch.zeros_like(cdf_l[:, :1]), cdf_l[:, :-1]], dim=-1)
                    prob_win = torch.sum(probs_w * p_l_less_than_i, dim=-1)
                    correct = (prob_win > 0.5).sum().item()
                    total_correct_predictions += correct
            
            avg_eval_loss = total_eval_loss / len(eval_dataloader)
            accuracy = total_correct_predictions / len(eval_dataset)
            print(f"Validation Loss: {avg_eval_loss:.4f} | Validation Accuracy: {accuracy:.4f}")
            
            # --- Сохранение чекпоинта ---
            checkpoint_dir = os.path.join(config.OUTPUT_DIR, f"checkpoint-step-{global_step}")
            os.makedirs(checkpoint_dir, exist_ok=True)
            print(f"Saving checkpoint to {checkpoint_dir}...")
            prob_rm_model.save_pretrained(checkpoint_dir)
            tokenizer.save_pretrained(checkpoint_dir)
            saved_checkpoints.append(checkpoint_dir)
            
            if len(saved_checkpoints) > config.NUM_CHECKPOINTS_TO_KEEP:
                dir_to_delete = saved_checkpoints.pop(0)
                print(f"Removing old checkpoint: {dir_to_delete}")
                shutil.rmtree(dir_to_delete)
            
            # НОВОЕ: Возвращаем модель в режим обучения
            prob_rm_model.train()
    
    # Печать в конце эпохи (можно оставить для общей информации)
    print(f"\n--- Epoch {epoch+1}/{config.NUM_EPOCHS} Finished ---")

# -- Шаг 2.6: Сохранение финальной модели --
print("\nTraining finished. Saving final model...")
final_model_dir = os.path.join(config.OUTPUT_DIR, "final_model")
os.makedirs(final_model_dir, exist_ok=True)
prob_rm_model.save_pretrained(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"Final model and tokenizer saved to {final_model_dir}")

Using device: cuda


Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at HuggingFaceTB/SmolLM2-135M-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Train dataset size: 6501
Validation dataset size: 723


Training Steps:   0%|          | 0/1626 [00:00<?, ?it/s]


--- Running Validation & Saving Checkpoint at Step 500 ---


...


--- Running Validation & Saving Checkpoint at Step 1000 ---


Validating at step 1000:   0%|          | 0/181 [00:00<?, ?it/s]

Validation Loss: 0.7333 | Validation Accuracy: 0.4260
Saving checkpoint to /trinity/home/a.anokhin/test_rl/test_task/probabilistic_rm_model/checkpoint-step-1000...

--- Running Validation & Saving Checkpoint at Step 1500 ---


Validating at step 1500:   0%|          | 0/181 [00:00<?, ?it/s]

Validation Loss: 0.7242 | Validation Accuracy: 0.4647
Saving checkpoint to /trinity/home/a.anokhin/test_rl/test_task/probabilistic_rm_model/checkpoint-step-1500...
Removing old checkpoint: /trinity/home/a.anokhin/test_rl/test_task/probabilistic_rm_model/checkpoint-step-500

--- Epoch 1/1 Finished ---

Training finished. Saving final model...
Final model and tokenizer saved to /trinity/home/a.anokhin/test_rl/test_task/probabilistic_rm_model/final_model


In [1]:
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from tqdm import tqdm
import numpy as np
import os

# --- 1. Конфигурация ---
SFT_MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
# <<< ИЗМЕНЕНИЕ 1: Укажите путь к вашей новой вероятностной RM >>>
# Убедитесь, что это папка, содержащая checkpoint (например, final_model или checkpoint-step-XXXX)
RM_PATH = "/trinity/home/a.anokhin/test_rl/test_task/probabilistic_rm_model/final_model" 
REINFORCE_SAVE_PATH = "./reinforce_model_smollm2_prob_rm"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Гиперпараметры для REINFORCE
LEARNING_RATE = 1e-6
NUM_EPOCHS = 1 
BATCH_SIZE = 4
MAX_PROMPTS = 2000
MAX_NEW_TOKENS = 128
EVAL_BATCH_SIZE = 8
EVAL_PROMPTS = 200

print(f"Используемое устройство: {DEVICE}")

# --- 2. Вспомогательный класс для Baseline ---
class MovingAverageBaseline:
    # ... (код без изменений) ...
    def __init__(self, momentum=0.9):
        self.momentum = momentum
        self.value = 0
        self.is_initialized = False

    def update(self, new_values):
        if not self.is_initialized:
            self.value = new_values.mean().item()
            self.is_initialized = True
        else:
            new_mean = new_values.mean().item()
            self.value = self.momentum * self.value + (1 - self.momentum) * new_mean
    
    def get(self):
        return self.value

# --- 3. Загрузка моделей и токенайзера ---
print("Загрузка моделей и токенайзера...")

# Policy Model (модель, которую мы будем обучать)
policy_model = AutoModelForCausalLM.from_pretrained(SFT_MODEL_ID).to(DEVICE)

# <<< ИЗМЕНЕНИЕ 2: Загружаем вероятностную Reward Model >>>
# AutoModelForSequenceClassification сам определит num_labels=10 из конфига модели
reward_model = AutoModelForSequenceClassification.from_pretrained(RM_PATH).to(DEVICE)
reward_model.eval() # Переводим в режим оценки

# Токенайзер
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_ID, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    policy_model.config.pad_token_id = policy_model.config.eos_token_id

# --- 4. Загрузка и подготовка данных ---
print("Загрузка и подготовка датасета...")
dataset = load_dataset("juyoungml/HelpSteer2-binarized", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_prompts = dataset["train"].select(range(MAX_PROMPTS))
val_prompts = dataset["test"].select(range(EVAL_PROMPTS))

def format_prompt(prompt_text):
    return f"<|user|>\n{prompt_text}<|end|>\n<|assistant|>\n"

# <<< ИЗМЕНЕНИЕ 3: Новая функция для расчета скалярной награды из распределения >>>
def calculate_expected_reward(logits: torch.Tensor) -> torch.Tensor:
    """
    Преобразует логиты из вероятностной RM в скалярную награду.
    Args:
        logits (torch.Tensor): Тензор с логитами формы [batch_size, num_classes]
    Returns:
        torch.Tensor: Тензор со скалярными наградами формы [batch_size]
    """
    if logits.dim() == 1: # Если на входе один пример, добавляем batch-измерение
        logits = logits.unsqueeze(0)
    
    num_classes = logits.shape[-1]
    
    # 1. Преобразуем логиты в вероятности
    probs = F.softmax(logits, dim=-1)
    
    # 2. Создаем тензор с "оценками" для каждой корзины (0, 1, 2, ..., 9)
    scores = torch.arange(num_classes, device=probs.device, dtype=probs.dtype)
    
    # 3. Считаем матожидание: sum(p_i * score_i) для каждого элемента в батче
    expected_reward = torch.sum(probs * scores, dim=-1)
    
    return expected_reward

# --- 5. Функция оценки ---
@torch.no_grad()
def evaluate_model(model, tokenizer, prompts, batch_size):
    model.eval()
    all_rewards = []
    
    print(f"Оценка модели на {len(prompts)} промптах...")
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts_text = prompts[i:i+batch_size]['prompt']
        formatted_prompts = [format_prompt(p) for p in batch_prompts_text]
        inputs = tokenizer(formatted_prompts, return_tensors="pt", padding=True).to(DEVICE)
        
        generated_outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id,
            do_sample=True, top_k=50, top_p=0.95
        )
        generated_texts = tokenizer.batch_decode(generated_outputs, skip_special_tokens=True)
        
        reward_inputs = tokenizer(
            generated_texts, return_tensors="pt", padding=True, 
            truncation=True, max_length=2048
        ).to(DEVICE)
        
        # <<< ИЗМЕНЕНИЕ 4: Используем новую логику расчета награды в оценке >>>
        reward_logits = reward_model(**reward_inputs).logits
        rewards = calculate_expected_reward(reward_logits)
        
        all_rewards.extend(rewards.cpu().tolist())
        
    return np.mean(all_rewards)

# --- 6. Основной цикл обучения REINFORCE ---
optimizer = torch.optim.Adam(policy_model.parameters(), lr=LEARNING_RATE)
baseline = MovingAverageBaseline(momentum=0.99)

sft_avg_reward = evaluate_model(policy_model, tokenizer, val_prompts, EVAL_BATCH_SIZE)
print(f"\nСредняя награда исходной SFT модели: {sft_avg_reward:.4f}")

policy_model.train()

print("\nНачинаем обучение с помощью REINFORCE w/ probabilistic RM...")
for epoch in range(NUM_EPOCHS):
    print(f"--- Эпоха {epoch + 1} / {NUM_EPOCHS} ---")
    
    shuffled_prompts = train_prompts.shuffle(seed=42+epoch)
    
    for i in tqdm(range(0, len(shuffled_prompts), BATCH_SIZE)):
        # 1. Формируем батч промптов
        batch_prompts_text = shuffled_prompts[i:i+BATCH_SIZE]['prompt']
        formatted_prompts = [format_prompt(p) for p in batch_prompts_text]
        prompt_inputs = tokenizer(formatted_prompts, return_tensors="pt", padding=True).to(DEVICE)
        prompt_len = prompt_inputs.attention_mask.size(1)

        # 2. Сэмплируем действия (генерируем ответы)
        generated_outputs = policy_model.generate(
            **prompt_inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id,
            do_sample=True, top_k=50, top_p=0.95, return_dict_in_generate=True, output_scores=True
        )
        
        # 3. Получаем награду (Reward) от Reward Model
        full_sequences_ids = generated_outputs.sequences
        full_sequences_text = tokenizer.batch_decode(full_sequences_ids, skip_special_tokens=True)
        
        reward_inputs = tokenizer(
            full_sequences_text, return_tensors="pt", padding=True, 
            truncation=True, max_length=2048
        ).to(DEVICE)
        
        with torch.no_grad():
            # <<< ИЗМЕНЕНИЕ 5: Используем новую логику расчета награды в цикле обучения >>>
            reward_logits = reward_model(**reward_inputs).logits
            rewards = calculate_expected_reward(reward_logits)

        # 4. Обновляем и получаем baseline
        baseline.update(rewards)
        b = baseline.get()
        
        # 5. Считаем Advantage
        advantages = rewards - b
        
        # 6. Считаем log(pi(a|s))
        generated_ids = full_sequences_ids[:, prompt_len:]
        full_logits = policy_model(full_sequences_ids).logits
        generated_logits = full_logits[:, prompt_len-1:-1, :]
        log_probs = F.cross_entropy(
            generated_logits.reshape(-1, generated_logits.size(-1)), 
            generated_ids.reshape(-1), 
            reduction='none'
        )
        log_probs = log_probs.reshape(generated_ids.size())
        mask = (generated_ids != tokenizer.pad_token_id).float()
        log_probs_sum = (log_probs * mask).sum(dim=1)
        
        # 7. Считаем loss для REINFORCE
        loss = (advantages * log_probs_sum).mean() # Минимизируем отрицательный loss
        
        if i % 50 == 0:
            print(f"\nШаг {i}/{len(shuffled_prompts)//BATCH_SIZE} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Mean Reward: {rewards.mean().item():.4f} | "
                  f"Baseline: {b:.4f}")
        
        # 8. Backpropagation и шаг оптимизатора
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
# --- 7. Финальная оценка и сохранение модели ---
print("\nОбучение завершено. Финальная оценка модели...")
reinforce_avg_reward = evaluate_model(policy_model, tokenizer, val_prompts, EVAL_BATCH_SIZE)
print(f"\nСредняя награда исходной SFT модели: {sft_avg_reward:.4f}")
print(f"Средняя награда REINFORCE модели (с вероятностной RM): {reinforce_avg_reward:.4f}")

if reinforce_avg_reward > sft_avg_reward:
    print("\n✅ Средняя награда на отложенной выборке выросла!")
else:
    print("\n⚠️ Средняя награда на отложенной выборке не выросла или уменьшилась.")

print(f"Сохранение REINFORCE модели в {REINFORCE_SAVE_PATH}...")
policy_model.save_pretrained(REINFORCE_SAVE_PATH)
tokenizer.save_pretrained(REINFORCE_SAVE_PATH)
print("Модель успешно сохранена.")

Используемое устройство: cuda
Загрузка моделей и токенайзера...
Загрузка и подготовка датасета...
Оценка модели на 200 промптах...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [01:31<00:00,  3.65s/it]



Средняя награда исходной SFT модели: 4.3431

Начинаем обучение с помощью REINFORCE w/ probabilistic RM...
--- Эпоха 1 / 1 ---


  0%|▏                                                                                                                      | 1/500 [00:03<32:23,  3.89s/it]


Шаг 0/500 | Loss: 24.4924 | Mean Reward: 3.6784 | Baseline: 3.6784


...


Шаг 300/500 | Loss: 60.1639 | Mean Reward: 4.6619 | Baseline: 3.9921


 20%|███████████████████████▋                                                                                             | 101/500 [06:08<23:45,  3.57s/it]


Шаг 400/500 | Loss: 113.0909 | Mean Reward: 4.8317 | Baseline: 4.0763


 22%|█████████████████████████▌                                                                                           | 109/500 [06:37<23:45,  3.65s/it]


KeyboardInterrupt: 

In [2]:
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from tqdm import tqdm
import numpy as np
import os

# --- 1. Конфигурация ---
SFT_MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
# <<< ИЗМЕНЕНИЕ 1: Укажите путь к вашей новой вероятностной RM >>>
# Убедитесь, что это папка, содержащая checkpoint (например, final_model или checkpoint-step-XXXX)
RM_PATH = "/trinity/home/a.anokhin/test_rl/test_task/probabilistic_rm_model/final_model" 
REINFORCE_SAVE_PATH = "./reinforce_model_smollm2_prob_rm"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Гиперпараметры для REINFORCE
LEARNING_RATE = 1e-6
NUM_EPOCHS = 1 
BATCH_SIZE = 4
MAX_PROMPTS = 2000
MAX_NEW_TOKENS = 128
EVAL_BATCH_SIZE = 8
EVAL_PROMPTS = 200
LAMBDA_PENALTY = 0.5

print(f"Используемое устройство: {DEVICE}")

# --- 2. Вспомогательный класс для Baseline ---
class MovingAverageBaseline:
    # ... (код без изменений) ...
    def __init__(self, momentum=0.9):
        self.momentum = momentum
        self.value = 0
        self.is_initialized = False

    def update(self, new_values):
        if not self.is_initialized:
            self.value = new_values.mean().item()
            self.is_initialized = True
        else:
            new_mean = new_values.mean().item()
            self.value = self.momentum * self.value + (1 - self.momentum) * new_mean
    
    def get(self):
        return self.value

# --- 3. Загрузка моделей и токенайзера ---
print("Загрузка моделей и токенайзера...")

# Policy Model (модель, которую мы будем обучать)
policy_model = AutoModelForCausalLM.from_pretrained(SFT_MODEL_ID).to(DEVICE)

# <<< ИЗМЕНЕНИЕ 2: Загружаем вероятностную Reward Model >>>
# AutoModelForSequenceClassification сам определит num_labels=10 из конфига модели
reward_model = AutoModelForSequenceClassification.from_pretrained(RM_PATH).to(DEVICE)
reward_model.eval() # Переводим в режим оценки

# Токенайзер
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_ID, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    policy_model.config.pad_token_id = policy_model.config.eos_token_id

# --- 4. Загрузка и подготовка данных ---
print("Загрузка и подготовка датасета...")
dataset = load_dataset("juyoungml/HelpSteer2-binarized", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_prompts = dataset["train"].select(range(MAX_PROMPTS))
val_prompts = dataset["test"].select(range(EVAL_PROMPTS))

def format_prompt(prompt_text):
    return f"<|user|>\n{prompt_text}<|end|>\n<|assistant|>\n"

# <<< ИЗМЕНЕНИЕ 3: Новая функция для расчета скалярной награды из распределения >>>
def calculate_expected_reward(logits: torch.Tensor) -> torch.Tensor:
    """
    Преобразует логиты из вероятностной RM в скалярную награду.
    Args:
        logits (torch.Tensor): Тензор с логитами формы [batch_size, num_classes]
    Returns:
        torch.Tensor: Тензор со скалярными наградами формы [batch_size]
    """
    if logits.dim() == 1: # Если на входе один пример, добавляем batch-измерение
        logits = logits.unsqueeze(0)
    
    num_classes = logits.shape[-1]
    
    # 1. Преобразуем логиты в вероятности
    probs = F.softmax(logits, dim=-1)
    
    # 2. Создаем тензор с "оценками" для каждой корзины (0, 1, 2, ..., 9)
    scores = torch.arange(num_classes, device=probs.device, dtype=probs.dtype)
    
    # 3. Считаем матожидание: sum(p_i * score_i) для каждого элемента в батче
    expected_reward = torch.sum(probs * scores, dim=-1)
    
    return expected_reward

def calculate_risk_adjusted_reward(logits: torch.Tensor, lambda_penalty: float) -> torch.Tensor:
    """
    Преобразует логиты из вероятностной RM в скалярную награду,
    штрафуя за высокую дисперсию.
    Reward = E[R] - lambda * Var[R]

    Args:
        logits (torch.Tensor): Тензор с логитами формы [batch_size, num_classes].
        lambda_penalty (float): Коэффициент неприятия риска. Чем больше, тем сильнее штраф.

    Returns:
        torch.Tensor: Тензор со скалярными наградами формы [batch_size].
    """
    if logits.dim() == 1:
        logits = logits.unsqueeze(0)
    
    num_classes = logits.shape[-1]
    probs = F.softmax(logits, dim=-1)
    
    # Создаем тензоры с оценками и их квадратами
    scores = torch.arange(num_classes, device=probs.device, dtype=probs.dtype)
    scores_squared = scores**2
    
    # Считаем E[R] (мат. ожидание)
    expected_reward = torch.sum(probs * scores, dim=-1)
    
    # Считаем E[R^2]
    expected_reward_squared = torch.sum(probs * scores_squared, dim=-1)
    
    # Считаем дисперсию: Var(R) = E[R^2] - (E[R])^2
    variance = expected_reward_squared - expected_reward**2
    
    # Итоговая награда с поправкой на риск
    risk_adjusted_reward = expected_reward - lambda_penalty * variance
    
    return risk_adjusted_reward

def sample_reward_from_distribution(logits: torch.Tensor) -> torch.Tensor:
    """
    Сэмплирует одну скалярную награду для каждого примера из распределения,
    заданного логитами от RM.

    Args:
        logits (torch.Tensor): Тензор с логитами формы [batch_size, num_classes].

    Returns:
        torch.Tensor: Тензор со скалярными наградами формы [batch_size].
    """
    if logits.dim() == 1:
        logits = logits.unsqueeze(0)
        
    probs = F.softmax(logits, dim=-1)
    
    # Сэмплируем один индекс (нашу оценку-награду) для каждого распределения в батче.
    # Результатом будет тензор формы [batch_size, 1] с индексами.
    sampled_indices = torch.multinomial(probs, num_samples=1)
    
    # Убираем лишнее измерение и приводим к типу float для дальнейших вычислений
    return sampled_indices.squeeze(-1).float()

# --- 5. Функция оценки ---
@torch.no_grad()
def evaluate_model(model, tokenizer, prompts, batch_size):
    model.eval()
    all_rewards = []
    
    print(f"Оценка модели на {len(prompts)} промптах...")
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts_text = prompts[i:i+batch_size]['prompt']
        formatted_prompts = [format_prompt(p) for p in batch_prompts_text]
        inputs = tokenizer(formatted_prompts, return_tensors="pt", padding=True).to(DEVICE)
        
        generated_outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id,
            do_sample=True, top_k=50, top_p=0.95
        )
        generated_texts = tokenizer.batch_decode(generated_outputs, skip_special_tokens=True)
        
        reward_inputs = tokenizer(
            generated_texts, return_tensors="pt", padding=True, 
            truncation=True, max_length=2048
        ).to(DEVICE)
        
        # <<< ИЗМЕНЕНИЕ 4: Используем новую логику расчета награды в оценке >>>
        reward_logits = reward_model(**reward_inputs).logits
        rewards = calculate_expected_reward(reward_logits)
        
        all_rewards.extend(rewards.cpu().tolist())
        
    return np.mean(all_rewards)

# --- 6. Основной цикл обучения REINFORCE ---
optimizer = torch.optim.Adam(policy_model.parameters(), lr=LEARNING_RATE)
baseline = MovingAverageBaseline(momentum=0.99)

sft_avg_reward = evaluate_model(policy_model, tokenizer, val_prompts, EVAL_BATCH_SIZE)
print(f"\nСредняя награда исходной SFT модели: {sft_avg_reward:.4f}")

policy_model.train()

print("\nНачинаем обучение с помощью REINFORCE w/ probabilistic RM...")
for epoch in range(NUM_EPOCHS):
    print(f"--- Эпоха {epoch + 1} / {NUM_EPOCHS} ---")
    
    shuffled_prompts = train_prompts.shuffle(seed=42+epoch)
    
    for i in tqdm(range(0, len(shuffled_prompts), BATCH_SIZE)):
        # 1. Формируем батч промптов
        batch_prompts_text = shuffled_prompts[i:i+BATCH_SIZE]['prompt']
        formatted_prompts = [format_prompt(p) for p in batch_prompts_text]
        prompt_inputs = tokenizer(formatted_prompts, return_tensors="pt", padding=True).to(DEVICE)
        prompt_len = prompt_inputs.attention_mask.size(1)

        # 2. Сэмплируем действия (генерируем ответы)
        generated_outputs = policy_model.generate(
            **prompt_inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id,
            do_sample=True, top_k=50, top_p=0.95, return_dict_in_generate=True, output_scores=True
        )
        
        # 3. Получаем награду (Reward) от Reward Model
        full_sequences_ids = generated_outputs.sequences
        full_sequences_text = tokenizer.batch_decode(full_sequences_ids, skip_special_tokens=True)
        
        reward_inputs = tokenizer(
            full_sequences_text, return_tensors="pt", padding=True, 
            truncation=True, max_length=2048
        ).to(DEVICE)
        
        with torch.no_grad():
            # <<< ИЗМЕНЕНИЕ 5: Используем новую логику расчета награды в цикле обучения >>>
            reward_logits = reward_model(**reward_inputs).logits
            #rewards = calculate_expected_reward(reward_logits)
            rewards = calculate_risk_adjusted_reward(reward_logits, lambda_penalty=LAMBDA_PENALTY)

        # 4. Обновляем и получаем baseline
        baseline.update(rewards)
        b = baseline.get()
        
        # 5. Считаем Advantage
        advantages = rewards - b
        
        # 6. Считаем log(pi(a|s))
        generated_ids = full_sequences_ids[:, prompt_len:]
        full_logits = policy_model(full_sequences_ids).logits
        generated_logits = full_logits[:, prompt_len-1:-1, :]
        log_probs = F.cross_entropy(
            generated_logits.reshape(-1, generated_logits.size(-1)), 
            generated_ids.reshape(-1), 
            reduction='none'
        )
        log_probs = log_probs.reshape(generated_ids.size())
        mask = (generated_ids != tokenizer.pad_token_id).float()
        log_probs_sum = (log_probs * mask).sum(dim=1)
        
        # 7. Считаем loss для REINFORCE
        loss = (advantages * log_probs_sum).mean() # Минимизируем отрицательный loss
        
        if i % 50 == 0:
            print(f"\nШаг {i}/{len(shuffled_prompts)//BATCH_SIZE} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Mean Reward: {rewards.mean().item():.4f} | "
                  f"Baseline: {b:.4f}")
        
        # 8. Backpropagation и шаг оптимизатора
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
# --- 7. Финальная оценка и сохранение модели ---
print("\nОбучение завершено. Финальная оценка модели...")
reinforce_avg_reward = evaluate_model(policy_model, tokenizer, val_prompts, EVAL_BATCH_SIZE)
print(f"\nСредняя награда исходной SFT модели: {sft_avg_reward:.4f}")
print(f"Средняя награда REINFORCE модели (с вероятностной RM): {reinforce_avg_reward:.4f}")

if reinforce_avg_reward > sft_avg_reward:
    print("\n✅ Средняя награда на отложенной выборке выросла!")
else:
    print("\n⚠️ Средняя награда на отложенной выборке не выросла или уменьшилась.")

print(f"Сохранение REINFORCE модели в {REINFORCE_SAVE_PATH}...")
policy_model.save_pretrained(REINFORCE_SAVE_PATH)
tokenizer.save_pretrained(REINFORCE_SAVE_PATH)
print("Модель успешно сохранена.")

Используемое устройство: cuda
Загрузка моделей и токенайзера...
Загрузка и подготовка датасета...
Оценка модели на 200 промптах...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [01:31<00:00,  3.64s/it]



Средняя награда исходной SFT модели: 4.4380

Начинаем обучение с помощью REINFORCE w/ probabilistic RM...
--- Эпоха 1 / 1 ---


  0%|▏                                                                                                               | 1/500 [00:03<29:16,  3.52s/it]


Шаг 0/500 | Loss: 7.1811 | Mean Reward: 1.9510 | Baseline: 1.9510


...


Обучение завершено. Финальная оценка модели...
Оценка модели на 200 промптах...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [01:31<00:00,  3.65s/it]


Средняя награда исходной SFT модели: 4.4380
Средняя награда REINFORCE модели (с вероятностной RM): 4.3921

⚠️ Средняя награда на отложенной выборке не выросла или уменьшилась.
Сохранение REINFORCE модели в ./reinforce_model_smollm2_prob_rm...
[2025-06-24 04:14:34,567] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)



/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


Модель успешно сохранена.
